# College Scorecard 
#### Goal: Identify ROI on Education by State, Institution and Major

Source: Department of Education College Scorecard Most Recent Cohorts_Institution Level and Field of Study data sets 

In [0]:
CREATE OR REPLACE TABLE project.default.bronze_institution AS 
SELECT * FROM project.default.most_recent_cohorts_institution

In [0]:
CREATE OR REPLACE TABLE project.default.bronze_field_of_study AS 
SELECT * FROM project.default.most_recent_cohorts_field_of_study

In [0]:
SELECT * FROM project.default.bronze_institution 

In [0]:
CREATE OR REPLACE TABLE project.default.silver_institution AS 
SELECT
  UNITID,
  STABBR AS State,
  LOCALE,
  LATITUDE,
  LONGITUDE, 
  INSTNM AS Institution 
FROM project.default.bronze_institution

In [0]:
CREATE OR REPLACE TABLE project.default.silver_field_of_study AS 
SELECT
  UNITID,
  CIPDESC AS Major,
  CONTROL AS School_type,
  CREDDESC AS Degree,
  -- Convert PS and NA to Null in debt and earnings columns, then filter out nulls
  CAST(NULLIF(NULLIF(DEBT_ALL_STGP_ANY_MDN, 'PS'), 'NA') AS FLOAT) AS Median_debt,
  CAST(NULLIF(NULLIF(EARN_MDN_5YR, 'PS'), 'NA') AS FLOAT) AS Median_earnings_5yr,
  -- Calculate Debt-to-earnings ratio
  CAST(NULLIF(NULLIF(DEBT_ALL_STGP_ANY_MDN, 'PS'), 'NA') AS FLOAT) / CAST(NULLIF(NULLIF(EARN_MDN_5YR, 'PS'), 'NA') AS FLOAT) AS DEBT_TO_EARNINGS_RATIO
FROM project.default.bronze_field_of_study
WHERE NULLIF(NULLIF(DEBT_ALL_STGP_ANY_MDN, 'PS'), 'NA') IS NOT NULL
  AND NULLIF(NULLIF(EARN_MDN_5YR, 'PS'), 'NA') IS NOT NULL

In [0]:
SELECT * FROM project.default.silver_field_of_study

In [0]:
SELECT DISTINCT Major FROM project.default.silver_field_of_study

In [0]:
CREATE OR REPLACE TABLE project.default.gold AS
SELECT 
  f.UNITID,
  i.Institution,
  i.State,
  -- Categorize states by region
  CASE
    WHEN i.State IN ('ME', 'NH', 'VT', 'MA', 'RI', 'CT', 'NY', 'NJ', 'PA') THEN 'Northeast'
    WHEN i.State IN ('DE', 'MD', 'DC', 'VA', 'WV', 'NC', 'SC', 'GA', 'FL', 'KY', 'TN', 'AL', 'MS', 'AR', 'LA') THEN 'Southeast'
    WHEN i.State IN ('OH', 'MI', 'IN', 'IL', 'WI', 'MN', 'IA', 'MO', 'ND', 'SD', 'NE', 'KS') THEN 'Midwest'
    WHEN i.State IN ('OK', 'TX', 'NM', 'AZ') THEN 'Southwest'
    WHEN i.State IN ('MT', 'ID', 'WY', 'CO', 'UT', 'NV', 'CA', 'OR', 'WA', 'AK', 'HI') THEN 'West'
    ELSE 'Other'
  END AS Region,
  i.LOCALE,
  i.LATITUDE,
  i.LONGITUDE,
  f.Major,
  -- Categorize majors using pattern matching
  CASE 
    WHEN f.Major LIKE '%Architecture%' THEN 'Architecture'
    WHEN f.Major LIKE '%Computer%' OR f.Major LIKE '%Information Technology%' OR f.Major LIKE '%Data%' THEN 'IT'
    WHEN f.Major LIKE '%Engineering%' THEN 'Engineering'
    WHEN f.Major LIKE '%Liberal Arts%' OR f.Major LIKE '%English%' OR f.Major LIKE '%Literature%' OR f.Major LIKE '%History%' OR f.Major LIKE '%Philosophy%' OR f.Major LIKE '%Language%' OR f.Major LIKE '%Classical%' OR f.Major LIKE '%Bible%' OR f.Major LIKE '%Religion%' OR f.Major LIKE '%Geography%' OR f.Major LIKE '%Diversity%' OR f.Major LIKE '%Pastoral%' OR f.Major LIKE '%Theological%' OR f.Major LIKE '%Museology%' OR f.Major LIKE '%Archeology%' OR f.Major LIKE '%Peace%' OR f.Major LIKE '%Historic%' OR f.Major LIKE '%Missions/Missionary%' THEN 'Humanities'
    WHEN f.Major LIKE '%Biology%' OR f.Major LIKE '%Marine%' OR f.Major LIKE '%Agriculture%' OR f.Major LIKE '%Agricultural%' OR f.Major LIKE '%Natural Resources%' OR f.Major LIKE '%Forestry%' OR f.Major LIKE '%Sustainability%' OR f.Major LIKE '%Genetics%' OR f.Major LIKE '%Food%' THEN 'Life Sciences'
    WHEN f.Major LIKE '%Biotechnology%' OR f.Major LIKE '%Astronomy%' OR f.Major LIKE '%Statistics%' OR f.Major LIKE '%Chemistry%' OR f.Major LIKE '%Physics%' OR f.Major LIKE '%Mathematics%' OR (f.Major LIKE '%Science%' AND f.Major NOT LIKE '%Liberal Arts%' AND f.Major NOT LIKE '%Social%' AND f.Major NOT LIKE '%Political%') THEN 'Hard Sciences'
    WHEN f.Major LIKE '%Psychology%' OR f.Major LIKE '%Sociology%' OR f.Major LIKE '%Anthropology%' OR f.Major LIKE '%Cultural%' OR f.Major LIKE '%Leisure%' OR f.Major LIKE '%Criminology%' OR f.Major LIKE '%Biopsychology%' THEN 'Social Sciences'
    WHEN f.Major LIKE '%Business%' OR f.Major LIKE '%Accounting%' OR f.Major LIKE '%Finance%' OR f.Major LIKE '%Marketing%' OR f.Major LIKE '%Management%' OR f.Major LIKE '%Economics%' OR f.Major LIKE '%Insurance%' OR f.Major LIKE '%Operations%' THEN 'Business & Economics'
    WHEN f.Major LIKE '%Nursing%' OR f.Major LIKE '%Health%' OR f.Major LIKE '%Medical%' OR f.Major LIKE '%Dental%' OR f.Major LIKE '%Clinical%' OR f.Major LIKE '%Gerontology%' OR f.Major LIKE '%Geriatrics%' OR f.Major LIKE 'Neurobiology%' OR f.Major LIKE '%Ophthalmic%' OR f.Major LIKE '%Pharmacology%' OR f.Major LIKE '%Rehabilitation%' THEN 'Healthcare'
    WHEN f.Major LIKE '%Social%' OR f.Major LIKE '%Human Development%' OR f.Major LIKE '%Family%' OR f.Major LIKE '%Human Services%' OR f.Major LIKE '%Community%' OR f.Major LIKE '%Somatic%' THEN 'Social Work'
    WHEN f.Major LIKE '%Veterinary%' OR f.Major LIKE '%Animal%' THEN 'Veterinary'
    WHEN f.Major LIKE '%Education%' OR f.Major LIKE '%Curriculum%' OR f.Major LIKE '%Teaching%' THEN 'Education'
    WHEN f.Major LIKE '%Political%' OR f.Major LIKE '%Public%' OR f.Major LIKE '%Urban%' OR f.Major LIKE '%Housing%' OR f.Major LIKE '%International%' OR f.Major LIKE '%Homeland%' THEN 'Public Policy'
    WHEN f.Major LIKE '%Art%' OR f.Major LIKE '%Music%' OR f.Major LIKE '%Drama%' OR f.Major LIKE '%Dance%' OR f.Major LIKE '%Theatre%' OR f.Major LIKE '%Design%' OR f.Major LIKE '%Rhetoric%' THEN 'Arts'
    WHEN f.Major LIKE '%Communication%' OR f.Major LIKE '%Journalism%' OR f.Major LIKE '%Media%' OR f.Major LIKE '%Public Relations%' THEN 'Communications'
    WHEN f.Major LIKE '%Carpenter%' OR f.Major LIKE '%Construction%' OR f.Major LIKE '%Mechanic%' OR f.Major LIKE '%Electrical%' OR f.Major LIKE '%Heating%' OR f.Major LIKE '%Precision%' OR f.Major LIKE '%Vehicle%' OR f.Major LIKE '%Industrial%' OR f.Major LIKE '%Environmental%' OR f.Major LIKE '%Cosmetology%' OR f.Major LIKE '%Fire%' OR f.Major LIKE '%Air%' OR f.Major LIKE '%Mining%' OR f.Major LIKE '%Apparel%' OR f.Major LIKE '%Ground Transportation%' OR f.Major LIKE '%Quality Control%' OR f.Major LIKE  '%Electromechanical%' OR f.Major LIKE '%Plumbing%' OR f.Major LIKE '%Energy%' OR f.Major LIKE '%Culinary%' OR f.Major LIKE '%Woodworking%' THEN 'Skilled Trades'
    WHEN f.Major LIKE '%Real Estate%' THEN 'Real Estate'
    WHEN f.Major LIKE '%Legal%' OR f.Major LIKE '%Paralegal%' OR f.Major LIKE '%Justice%' THEN 'Legal'
    ELSE 'General/Undecided'
  END AS Major_Category,
  f.School_type,
  f.Degree,
  f.Median_debt,
  f.Median_earnings_5yr,
  f.DEBT_TO_EARNINGS_RATIO
FROM project.default.silver_field_of_study f
INNER JOIN project.default.silver_institution i
  ON f.UNITID = CAST(i.UNITID AS STRING)

In [0]:
SELECT * FROM project.default.gold

### Exploratory Data Analysis

In [0]:
SELECT
  State,
  AVG(CAST(LATITUDE AS FLOAT)) AS Avg_Latitude,
  AVG(CAST(LONGITUDE AS FLOAT)) AS Avg_Longitude,
  SUM(Median_debt) / SUM(Median_earnings_5yr) AS Debt_To_Earnings_Ratio
FROM project.default.gold
GROUP BY State
ORDER BY Debt_To_Earnings_Ratio DESC
LIMIT 5

In [0]:
SELECT
  Major_Category,
  SUM(Median_debt) / SUM(Median_earnings_5yr) AS Debt_To_Earnings_Ratio
FROM
  project.default.gold
GROUP BY
  Major_Category
ORDER BY
  Debt_To_Earnings_Ratio DESC

In [0]:
SELECT
Median_debt,
Median_earnings_5yr 
FROM project.default.gold 

In [0]:
SELECT
  School_type,
  SUM(Median_debt) / SUM(Median_earnings_5yr) AS Debt_To_Earnings_Ratio
FROM
  project.default.gold
GROUP BY
  School_type
ORDER BY
  Debt_To_Earnings_Ratio DESC

In [0]:
SELECT
  Region,
  School_type,
  SUM(Median_debt) / SUM(Median_earnings_5yr) AS Debt_To_Earnings_Ratio
FROM project.default.gold
GROUP BY Region, School_type
ORDER BY Region, Debt_To_Earnings_Ratio DESC